# Train in Mobilenetv3

In [10]:
import torch
from torch import nn, optim
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader
import os


In [11]:

# Paths
train_dir = "transform/train"
val_dir = "transform/val"


In [12]:

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [13]:
# Training transforms (includes augmentation)
train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),        # random flip
    transforms.RandomRotation(15),           # random rotation
    transforms.ToTensor(),                    # convert PIL to tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Validation transforms (no augmentation)
val_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [14]:
# Datasets and loaders
train_dataset = datasets.ImageFolder(train_dir, transform=train_transforms)
val_dataset = datasets.ImageFolder(val_dir, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)


In [15]:

# Model: MobileNetV3 Small pretrained
model = models.mobilenet_v3_small(Weights=True)

In [16]:

# Replace classifier for your dataset (3 classes: 0,1,2)
num_classes = 3
model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)
model = model.to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)


In [17]:

# Training loop
num_epochs = 16

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)

    # Validation
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    val_acc = correct / total

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Loss: {epoch_loss:.4f} "
          f"Val Acc: {val_acc:.4f}")


Epoch [1/16] Loss: 1.0020 Val Acc: 0.6436
Epoch [2/16] Loss: 0.6888 Val Acc: 0.6436
Epoch [3/16] Loss: 0.6446 Val Acc: 0.6436
Epoch [4/16] Loss: 0.5082 Val Acc: 0.6436
Epoch [5/16] Loss: 0.5544 Val Acc: 0.6436
Epoch [6/16] Loss: 0.4665 Val Acc: 0.6436
Epoch [7/16] Loss: 0.4000 Val Acc: 0.6436
Epoch [8/16] Loss: 0.3438 Val Acc: 0.6436
Epoch [9/16] Loss: 0.3212 Val Acc: 0.6436
Epoch [10/16] Loss: 0.4065 Val Acc: 0.6436
Epoch [11/16] Loss: 0.3589 Val Acc: 0.6436
Epoch [12/16] Loss: 0.2740 Val Acc: 0.6436
Epoch [13/16] Loss: 0.3749 Val Acc: 0.6436
Epoch [14/16] Loss: 0.4187 Val Acc: 0.6436
Epoch [15/16] Loss: 0.2915 Val Acc: 0.6436
Epoch [16/16] Loss: 0.3059 Val Acc: 0.6436


In [18]:

# Save model
torch.save(model.state_dict(), "model/mobilenetv3_fundus.pth")
print("✅ Model saved as mobilenetv3_fiqa.pth")


✅ Model saved as mobilenetv3_fiqa.pth
